#**CODE DỰA VÀO PAPER**
#####không chạy được 🤡

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor


# A. Data Collection
df = pd.read_csv("bengaluru_house_prices.csv")

# B. Dropping features
# Paper: "The following features: 'area_type', 'society', 'balcony', and 'availability' are dropped"
df = df.drop(['area_type', 'society', 'balcony', 'availability'], axis='columns')

# C. Handling Missing Values
# Paper: "You can either remove rows with missing values, fill them using interpolation, or use more advanced imputation techniques."
# Áp dụng nội suy tuyến tính (interpolation) và xóa các dòng rỗng còn sót lại như paper gợi ý.
df = df.interpolate(method='linear', limit_direction='forward')
df = df.dropna()

# D. Feature engineering
# Paper: "adding a new feature with the name 'price per square feet'."
# (Tác giả không hề nói phải chuyển đổi cột total_sqft từ chữ "1000 - 1200" sang số, nên để nguyên rồi tính)
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft']

# E. Dimensionality reduction
# Paper: "lower the number of features... By modifying or picking features"
# (Không có hướng dẫn cụ thể, nên dùng phương pháp nhóm các location xuất hiện < 10 lần thành 'other' - kỹ thuật chuẩn mực nhất cho khái niệm này).
df.location = df.location.apply(lambda x: str(x).strip())
location_stats = df.groupby('location')['location'].agg('count').sort_values(ascending=False)
location_stats_less_than_10 = location_stats[location_stats <= 10]
df.location = df.location.apply(lambda x: 'other' if x in location_stats_less_than_10 else x)

# F. Outlier removal
# Paper: "A common and simple technique of removing outliers are used here i.e standard deviation and mean."
# (Tuyệt nhiên không có dòng nào nhắc đến việc lọc ngoại lệ theo BHK hay giới hạn 300 sqft)
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)

# G. One-hot Encoding
# Paper: "one-hot encoding is a helpful step for the 'Location' feature"
dummies = pd.get_dummies(df.location)
df = pd.concat([df, dummies.drop('other', axis='columns')], axis='columns')
df = df.drop(['location', 'price_per_sqft'], axis='columns')

# Xác định biến phụ thuộc và độc lập
# Paper: "independent variables comprising the number of bedrooms, bathrooms in the house, square footage area..."
# (Paper không nói phải trích xuất số từ cột 'size' chứa chữ "2 BHK", nên giữ nguyên cột 'size' đưa vào mô hình)
X = df.drop('price', axis='columns')
y = df.price

# Phân chia tập dữ liệu: 80% training và 20% testing (Theo mục V. Conclusion)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

# H, I, J, K. Models & GridSearchCV
# Tinh chỉnh siêu tham số và kiểm tra chéo cho Linear Regression, Lasso, và Decision Tree
algos = {
    'linear_regression': {'model': LinearRegression(), 'params': {'fit_intercept': [True, False]}},
    'lasso': {'model': Lasso(), 'params': {'alpha': [1, 2], 'selection': ['random', 'cyclic']}},
    'decision_tree': {'model': DecisionTreeRegressor(), 'params': {'criterion': ['squared_error', 'friedman_mse'], 'splitter': ['best', 'random']}}
}

scores = []
print("Đang huấn luyện và tìm kiếm mô hình tốt nhất (GridSearchCV)...")

for algo_name, config in algos.items():
    gs = GridSearchCV(config['model'], config['params'], cv=5, return_train_score=False)
    gs.fit(X_train, y_train)
    scores.append({
        'model': algo_name,
        'best_score': gs.best_score_,
        'best_params': gs.best_params_
    })

# IV. RESULTS
results_df = pd.DataFrame(scores, columns=['model', 'best_score', 'best_params'])
print(results_df)

#**CODE ĐÃ SỬA**
#####Có thể đạt kết quả giống paper nhưng hơi bổ sung

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor


# A. Data Collection
try:
    df = pd.read_csv("/content/drive/MyDrive/Project/Bengaluru_House_Data.csv")
except FileNotFoundError:
    print("Vui lòng kiểm tra lại đường dẫn file CSV.")
    exit()

# B. Dropping features
df = df.drop(['area_type', 'society', 'balcony', 'availability'], axis='columns')

# C. Handling Missing Values
df = df.dropna()

# Xử lý cột 'size'
df['bhk'] = df['size'].apply(lambda x: int(x.split(' ')[0]))
df = df.drop(['size'], axis='columns')

# Tiền xử lý 'total_sqft'
def convert_sqft_to_num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df = df.dropna()

# D. Feature Engineering
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft']

# E. Dimensionality Reduction
df.location = df.location.apply(lambda x: x.strip())
location_stats = df.groupby('location')['location'].agg('count').sort_values(ascending=False)
location_stats_less_than_10 = location_stats[location_stats <= 10]
df.location = df.location.apply(lambda x: 'other' if x in location_stats_less_than_10 else x)

# F. Outlier removal

# --- MỚI THÊM: Lọc ngoại lệ diện tích phòng ngủ phi thực tế (< 300 sqft/phòng) ---
df = df[~(df.total_sqft/df.bhk < 300)]
# ----------------------------------------------------------------------------------

# 1. Lọc ngoại lệ độ lệch chuẩn và giá trị trung bình
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)

# 2. Lọc ngoại lệ BHK
def remove_bhk_outliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count'] > 5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft < (stats['mean'])].index.values)
    return df.drop(exclude_indices, axis='index')

df = remove_bhk_outliers(df)

# 3. Lọc ngoại lệ phòng tắm
df = df[df.bath < df.bhk + 2]

# G. One-hot Encoding
dummies = pd.get_dummies(df.location)
df = pd.concat([df, dummies.drop('other', axis='columns')], axis='columns')
df = df.drop(['location', 'price_per_sqft'], axis='columns')

# Xác định biến phụ thuộc và độc lập
X = df.drop('price', axis='columns')
y = df.price

# Phân chia tập dữ liệu: 80% training và 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

# H, I, J, K. Models & GridSearchCV
algos = {
    'linear_regression': {
        'model': LinearRegression(),
        'params': {
            'fit_intercept': [True, False]
        }
    },
    'lasso': {
        'model': Lasso(),
        'params': {
            'alpha': [1, 2],
            'selection': ['random', 'cyclic']
        }
    },
    'decision_tree': {
        'model': DecisionTreeRegressor(),
        'params': {
            'criterion': ['squared_error', 'friedman_mse'],
            'splitter': ['best', 'random']
        }
    }
}

scores = []
print("Đang huấn luyện và tìm kiếm mô hình tốt nhất (GridSearchCV)...")

for algo_name, config in algos.items():
    gs = GridSearchCV(config['model'], config['params'], cv=5, return_train_score=False)
    gs.fit(X_train, y_train)

    # Chấm điểm trên tập Test (20%) để lấy kết quả thực tế
    best_model = gs.best_estimator_
    test_score = best_model.score(X_test, y_test)

    scores.append({
        'model': algo_name,
        'test_score': test_score,
        'best_params': gs.best_params_
    })

# IV. RESULTS
results_df = pd.DataFrame(scores, columns=['model', 'test_score', 'best_params'])
print("\nBảng đánh giá Model Score (Đã hoàn thiện):")
print(results_df)

Đang huấn luyện và tìm kiếm mô hình tốt nhất (GridSearchCV)...

Bảng đánh giá Model Score (Đã hoàn thiện):
               model  test_score  \
0  linear_regression    0.862962   
1              lasso    0.718480   
2      decision_tree    0.711665   

                                         best_params  
0                           {'fit_intercept': False}  
1                {'alpha': 1, 'selection': 'cyclic'}  
2  {'criterion': 'squared_error', 'splitter': 'ra...  
